# Swin-UMamba trên Kaggle — Train / Val / Inference

Notebook này chạy lại pipeline của bài gốc *Swin-UMamba* (nnU-Net v2, cấu hình **2d**), nhưng dùng bộ dữ liệu mới.

**Trước khi chạy, cấu hình Kaggle:**
1. *Settings → Accelerator*: **GPU T4** (dùng 1 GPU). Không nên dùng P100 vì torch mới trên Kaggle không còn hỗ trợ tốt.
2. *Settings → Internet*: **On** (để clone GitHub, cài `mamba-ssm` và tải pretrained VMamba).
3. *Add Input*: thêm Kaggle Dataset chứa thư mục **BTXRD** (gồm `images/` và `Annotations/`). Notebook sẽ tự tìm thư mục này trong `/kaggle/input`, hoặc trong `data/BTXRD` của repo nếu đã đẩy lên GitHub.
4. Nếu train lâu hơn 12 giờ: dùng **Save Version → Save & Run All (Commit)**. Kết quả trong `/kaggle/working` sẽ được lưu lại. Ở phiên sau, thêm output đó làm input và đặt `RESUME_DIR` để train tiếp.

Các bước: clone repo → cài môi trường → tải pretrained → chuyển dữ liệu sang định dạng nnU-Net → `nnUNetv2_plan_and_preprocess` → `nnUNetv2_train` (kèm validation) → `nnUNetv2_predict` trên tập test → tính DSC / NSD → trực quan hoá.

## 0. Cấu hình

In [ ]:
import os, glob

# ---------- Mã nguồn ----------
REPO_URL = "https://github.com/67htb5vn/Swin-UMamba.git"
BRANCH   = "main"
WORK     = "/kaggle/working"
REPO     = f"{WORK}/Swin-UMamba"
TMP      = "/tmp/nnunet"            # raw + preprocessed để ở /tmp (đĩa lớn, không bị lưu vào output)

# ---------- Dữ liệu ----------
# "btxrd"     : chuyển BTXRD (LabelMe json) sang nnU-Net
# "nnunet_raw": dùng sẵn một thư mục DatasetXXX_* đúng định dạng nnU-Net (vd. dữ liệu gốc của U-Mamba)
DATA_SOURCE  = "btxrd"
BTXRD_DIR    = None                 # None = tự tìm trong /kaggle/input; hoặc vd "/kaggle/input/btxrd/BTXRD"
NNUNET_RAW_DIR = None               # dùng khi DATA_SOURCE="nnunet_raw", vd "/kaggle/input/umamba-data/Dataset704_Endovis17"
DATASET_ID   = 705
DATASET_NAME = f"Dataset{DATASET_ID:03d}_BTXRD"
LABEL_MODE   = "benign_malignant"   # "benign_malignant": nền(0)/u lành(1)/u ác(2) | "binary": nền/khối u
MAX_SIZE     = 1024                 # resize cạnh dài nhất về <= MAX_SIZE (ảnh X-quang gốc rất lớn); None = giữ nguyên
SPLIT_RATIO  = (0.80, 0.15, 0.05)   # train : val : test, chia stratified theo u lành / u ác
SEED         = 42

# ---------- Huấn luyện ----------
# Trainer dạng *Kaggle = trainer gốc + cho phép chỉnh epoch/iter qua biến môi trường (xem bước 4)
#   nnUNetTrainerSwinUMambaKaggle        : Swin-UMamba, pretrain ImageNet
#   nnUNetTrainerSwinUMambaDKaggle       : Swin-UMamba†, pretrain ImageNet
#   nnUNetTrainerSwinUMambaScratchKaggle : Swin-UMamba, không pretrain
#   nnUNetTrainerSwinUMambaDScratchKaggle: Swin-UMamba†, không pretrain
TRAINER         = "nnUNetTrainerSwinUMambaKaggle"
FOLD            = "0"               # giữ "0": fold 0 = đúng tập train/val 80:15 ở trên (ghi trong splits_final.json)
EPOCHS          = 100               # code gốc mặc định 1000 epoch -> quá dài cho Kaggle
ITERS_PER_EPOCH = 250
SAVE_EVERY      = 10                # lưu checkpoint_latest mỗi N epoch (để resume)
FREEZE_EPOCHS   = None              # số epoch đầu đóng băng encoder pretrained. Gốc = 10; None = tự tính min(10, EPOCHS//10)
BATCH_SIZE      = 4                 # T4 16GB: batch nnU-Net tự chọn quá lớn -> CUDA OOM. Vẫn OOM: giảm 2 hoặc MAX_SIZE=512
RESUME_DIR      = None              # vd "/kaggle/input/<output-phien-truoc>/Swin-UMamba/data/nnUNet_results"

# ---------- Cache môi trường ----------
WHEEL_DIR = None   # None = tự tìm file mamba_ssm-*.whl trong /kaggle/input; hoặc chỉ rõ, vd "/kaggle/input/swin-umamba-wheels"

PRETRAIN_URL = "https://github.com/MzeroMiko/VMamba/releases/download/%23v0cls/vssmtiny_dp01_ckpt_epoch_292.pth"

# ---------- Biến môi trường (được kế thừa bởi các lệnh `!`) ----------
os.environ.update({
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",   # torch>=2.6 mặc định weights_only=True -> không load được checkpoint nnU-Net
    "nnUNet_n_proc_DA": "4",                   # Kaggle có 4 CPU
    "nnUNet_compile": "f",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",   # giảm phân mảnh bộ nhớ GPU
    "MPLBACKEND": "Agg",
    "SUM_EPOCHS": str(EPOCHS),
    "SUM_ITERS": str(ITERS_PER_EPOCH),
    "SUM_SAVE_EVERY": str(SAVE_EVERY),
    "SUM_FREEZE_EPOCHS": str(FREEZE_EPOCHS if FREEZE_EPOCHS is not None else min(10, EPOCHS // 10)),
})

## 1. Kiểm tra GPU & clone repo từ GitHub

In [ ]:
!nvidia-smi
import subprocess
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=False)
!git -C {REPO} log --oneline -1

## 2. Cài môi trường (có cache wheel)
Bước chậm nhất là build `causal-conv1d` và `mamba-ssm` (biên dịch CUDA, có thể mất 20–40 phút). Ô dưới đây xử lý như sau:
- **Lần đầu:** build hai gói này thành file wheel và lưu vào `/kaggle/working/swin_umamba_wheels/`, kèm file `env_tag.txt` ghi phiên bản torch/CUDA/Python.
- **Các lần sau:** notebook tự tìm file `mamba_ssm-*.whl` và `causal_conv1d-*.whl` ở bất kỳ đâu trong `/kaggle/input` (hoặc trong `WHEEL_DIR` nếu bạn đặt), cài thẳng từ wheel rồi chạy thử kernel CUDA. Nếu wheel không còn tương thích (Kaggle đã đổi phiên bản torch), notebook tự build lại.

**Cách lưu cache để dùng lại** (chỉ cần làm 1 lần, sau khi lần chạy đầu thành công):
1. *Save Version* để lưu notebook, mở version vừa lưu, vào tab **Output**, chọn **New Dataset**, đặt tên ví dụ `swin-umamba-wheels`.
2. Ở các lần chạy sau, *Add Input* thêm dataset đó vào notebook. Cách upload nào cũng được: cả thư mục, hay chỉ 3 file `.whl` và `env_tag.txt`.
3. Nên bật *Settings → Environment → **Pin to original environment***. Kaggle cập nhật image định kỳ; nếu phiên bản torch đổi, wheel cũ không dùng được nữa và notebook sẽ tự build lại.

Các thư viện thuần Python còn lại (nnU-Net, monai, timm, …) vẫn cài từ PyPI như bình thường, chỉ mất khoảng 1–2 phút.

In [ ]:
import sys, torch
ENV_TAG = f"torch{torch.__version__.split('+')[0]}-cu{torch.version.cuda}-py{sys.version_info.major}{sys.version_info.minor}"
print("ENV_TAG:", ENV_TAG)

def find_wheel_cache():
    """Tìm mọi thư mục chứa đủ 2 wheel (trong WHEEL_DIR, /kaggle/input hoặc /kaggle/working), ưu tiên thư mục có env_tag khớp."""
    roots = [WHEEL_DIR] if WHEEL_DIR else []
    roots += sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/mamba_ssm-*.whl", recursive=True)})
    roots += [f"{WORK}/swin_umamba_wheels"]
    cands = []
    for d in roots:
        if not (d and glob.glob(f"{d}/mamba_ssm-*.whl") and glob.glob(f"{d}/causal_conv1d-*.whl")):
            continue
        tag_file = f"{d}/env_tag.txt"
        tag = open(tag_file).read().strip() if os.path.isfile(tag_file) else "?"
        print(f"  tìm thấy wheel: {d}  (env_tag={tag})")
        cands.append((tag != ENV_TAG, d))
    return sorted(cands)[0][1] if cands else None

def mamba_ok():
    test = ("import torch; from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
            "u=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); B=torch.randn(1,8,16,device='cuda'); "
            "selective_scan_fn(u,u,A,B,B)")
    return subprocess.run(["python", "-c", test]).returncode == 0

%cd {REPO}/swin_umamba
!pip install -q -e .
# setup.py ghim monai==1.3.0 (không tương thích numpy 2 trên Kaggle) -> dùng 1.4.0, API các block vẫn giữ nguyên
!pip install -q "monai==1.4.0" timm torchinfo einops numba packaging ninja

WHEEL_CACHE = find_wheel_cache()
ok = False
if WHEEL_CACHE:
    print("Cài mamba từ cache:", WHEEL_CACHE)
    !pip install -q --no-deps --force-reinstall {WHEEL_CACHE}/causal_conv1d-*.whl {WHEEL_CACHE}/mamba_ssm-*.whl
    ok = mamba_ok()   # wheel build với torch khác phiên bản sẽ lỗi ở đây -> tự build lại
    print("cache dùng được" if ok else "cache KHÔNG tương thích với torch hiện tại -> build lại")
if not ok:
    print("Không có cache khớp -> build wheel (lần đầu sẽ lâu)...")
    WHEEL_OUT = f"{WORK}/swin_umamba_wheels"
    !rm -rf {WHEEL_OUT} && mkdir -p {WHEEL_OUT}
    !pip wheel -q --no-build-isolation --no-deps causal-conv1d mamba-ssm -w {WHEEL_OUT}
    !pip install -q --no-deps {WHEEL_OUT}/causal_conv1d-*.whl {WHEEL_OUT}/mamba_ssm-*.whl
    open(f"{WHEEL_OUT}/env_tag.txt", "w").write(ENV_TAG)
    !ls -la {WHEEL_OUT}
%cd {REPO}

In [ ]:
# Kiểm tra kernel CUDA selective_scan của mamba
check = ("import torch, mamba_ssm; "
         "from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
         "u=torch.randn(1,4,16,device='cuda'); d=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); "
         "B=torch.randn(1,8,16,device='cuda'); C=torch.randn(1,8,16,device='cuda'); "
         "print('torch', torch.__version__, '| mamba_ssm', mamba_ssm.__version__, '| out', selective_scan_fn(u,d,A,B,C).shape)")
subprocess.run(["python", "-c", check], check=True)
!python -c "import nnunetv2, monai; from nnunetv2.paths import nnUNet_raw; print('nnUNet_raw =', nnUNet_raw)"

## 3. Tải pretrained VMamba-Tiny (ImageNet) & chuẩn bị thư mục dữ liệu

In [ ]:
os.chdir(REPO)
ckpt = f"{REPO}/data/pretrained/vmamba/vmamba_tiny_e292.pth"
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
import urllib.request
# file hợp lệ ~87 MB; file rỗng/hỏng (do lần tải trước bị lỗi) thì xoá và tải lại
if os.path.isfile(ckpt) and os.path.getsize(ckpt) < 50 * 2**20:
    os.remove(ckpt)
if not os.path.isfile(ckpt):
    urllib.request.urlretrieve(PRETRAIN_URL, ckpt + ".part")   # lỗi HTTP (404...) sẽ báo ngay, không tạo file rỗng
    os.rename(ckpt + ".part", ckpt)
assert os.path.getsize(ckpt) > 50 * 2**20, "File pretrained không hợp lệ"
print("pretrained:", ckpt, os.path.getsize(ckpt) // 2**20, "MB")

# raw & preprocessed nằm ở /tmp (symlink vào repo/data), results nằm trong /kaggle/working (được lưu lại)
RAW_DS = f"{TMP}/nnUNet_raw/{DATASET_NAME}"
os.makedirs(RAW_DS, exist_ok=True)
os.makedirs(f"{TMP}/nnUNet_preprocessed", exist_ok=True)
os.makedirs(f"{REPO}/data/nnUNet_results", exist_ok=True)
for link, target in [(f"{REPO}/data/nnUNet_raw/{DATASET_NAME}", RAW_DS),
                     (f"{REPO}/data/nnUNet_preprocessed", f"{TMP}/nnUNet_preprocessed")]:
    if not os.path.lexists(link):
        os.symlink(target, link)

if RESUME_DIR:   # khôi phục kết quả phiên trước để train tiếp
    !cp -rn {RESUME_DIR}/. {REPO}/data/nnUNet_results/
    print("restored results from", RESUME_DIR)

## 4. Trainer cho Kaggle
Các trainer này kế thừa nguyên trainer gốc (cùng AdamW, lr 1e-4, cosine schedule, đóng băng encoder 10 epoch đầu, deep supervision), chỉ thay đổi:
- `num_epochs`, `num_iterations_per_epoch`, `save_every`, `freeze_encoder_epochs`: đọc từ biến môi trường. Số epoch đóng băng encoder co giãn theo `EPOCHS`, để encoder vẫn được train khi `EPOCHS` nhỏ;
- không lưu `checkpoint_{epoch}.pth` (mỗi file khoảng 0.5 GB, dễ làm đầy 20 GB của `/kaggle/working`). Vẫn giữ `checkpoint_latest`, `checkpoint_best` và `checkpoint_final`.

In [ ]:
trainer_src = '''
import os
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMamba import nnUNetTrainerSwinUMamba
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaD import nnUNetTrainerSwinUMambaD
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaScratch import nnUNetTrainerSwinUMambaScratch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaDScratch import nnUNetTrainerSwinUMambaDScratch


def _kaggle_setup(self):
    self.num_epochs = int(os.environ.get("SUM_EPOCHS", 100))
    self.num_iterations_per_epoch = int(os.environ.get("SUM_ITERS", 250))
    self.save_every = int(os.environ.get("SUM_SAVE_EVERY", 10))
    # gốc: đóng băng encoder 10 epoch đầu -> nếu EPOCHS nhỏ thì encoder không bao giờ được train
    self.freeze_encoder_epochs = int(os.environ.get("SUM_FREEZE_EPOCHS", 10))

'''
for base in ["nnUNetTrainerSwinUMamba", "nnUNetTrainerSwinUMambaD",
             "nnUNetTrainerSwinUMambaScratch", "nnUNetTrainerSwinUMambaDScratch"]:
    trainer_src += f'''
class {base}Kaggle({base}):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 unpack_dataset: bool = True, device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        _kaggle_setup(self)

    def on_epoch_end(self):
        nnUNetTrainer.on_epoch_end(self)   # bỏ qua việc lưu checkpoint_{{epoch}}.pth của lớp cha

'''
path = f"{REPO}/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUMambaKaggle.py"
open(path, "w").write(trainer_src)
!python -c "from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaKaggle import *; print('OK')"

## 5. Chuẩn bị dữ liệu theo định dạng nnU-Net
**BTXRD:**
- **Chỉ dùng ảnh có u xương**, tức ảnh có nhãn **polygon** vùng u. Ảnh bình thường (không có annotation) và các nhãn `rectangle` (bbox) bị bỏ qua.
- **Nhãn mask** (`LABEL_MODE="benign_malignant"`): 0 = nền, 1 = **u lành**, 2 = **u ác**. U ác gồm `osteosarcoma` và `other mt`; các loại còn lại là u lành.
- **Nếu input là thư mục đã chia sẵn** `BTXRD_split/{train,val,test}/{images,Annotations}`, notebook dùng nguyên cách chia đó và bỏ qua `SPLIT_RATIO`. Notebook ưu tiên tìm thư mục dạng này trong `/kaggle/input`.
- Nếu không có thư mục chia sẵn: **chia train : val : test = 80 : 15 : 5** (`SPLIT_RATIO`), stratified để tỉ lệ u lành / u ác ở cả 3 tập đều như nhau. Một ảnh được tính là u ác nếu có ít nhất một polygon u ác.
  - train + val → `imagesTr/labelsTr`. Danh sách train/val được ghi vào `split.json`, và ở bước 6 sẽ được chép thành `splits_final.json`, để nnU-Net dùng đúng tập val này thay vì tự chia 5-fold.
  - test → `imagesTs/labelsTs`, chỉ dùng cho inference ở bước 8–9.
- Ảnh được chuyển sang PNG xám 1 kênh và resize cạnh dài về `MAX_SIZE`.
- Nếu đổi `LABEL_MODE`, `SPLIT_RATIO`, `MAX_SIZE` hoặc `SEED`, notebook tự chuyển đổi lại dữ liệu và preprocess lại.

In [ ]:
import json
convert_src = r"""
import os, sys, json, glob, random
import numpy as np, cv2
from PIL import Image, ImageOps

src, out, mode, max_size, ratios, seed, name = sys.argv[1:8]
max_size = int(max_size) if max_size != "None" else None
r_tr, r_va, r_te = [float(x) for x in ratios.split(",")]
seed = int(seed)

MALIGNANT = {"osteosarcoma", "other mt"}
# Nếu có split.csv (image_id,split,type) thì loại u của ảnh lấy theo cột type (benign/malignant),
# ngược lại suy ra từ tên nhãn polygon
import csv
TYPES = {}
for p in (os.path.join(src, "split.csv"), os.path.join(os.path.dirname(src), "split.csv")):
    if os.path.isfile(p):
        for r in csv.DictReader(open(p, encoding="utf-8-sig")):
            if r.get("image_id") and (r.get("type") or "").strip().lower() in ("benign", "malignant"):
                TYPES[os.path.splitext(r["image_id"].strip())[0]] = r["type"].strip().lower()
        print("Loại u lấy từ", p, "-", len(TYPES), "ảnh")
        break
def is_mal(stem, label):
    return TYPES[stem] == "malignant" if stem in TYPES else label.strip().lower() in MALIGNANT

if mode == "binary":
    labels = {"background": 0, "tumor": 1}
    cls = lambda stem, l: 1
else:
    labels = {"background": 0, "benign": 1, "malignant": 2}
    cls = lambda stem, l: 2 if is_mal(stem, l) else 1

def load_cases(root):
    imgs = {os.path.splitext(f)[0]: os.path.join(root, "images", f) for f in os.listdir(os.path.join(root, "images"))}
    out_cases = []
    for jf in sorted(glob.glob(os.path.join(root, "Annotations", "*.json"))):
        stem = os.path.splitext(os.path.basename(jf))[0]
        ann = json.load(open(jf, encoding="utf-8"))
        polys = [s for s in ann["shapes"] if s.get("shape_type") == "polygon" and len(s["points"]) >= 3]
        if stem in imgs and polys:
            out_cases.append((stem, imgs[stem], ann, polys))
    return out_cases

# Thư mục đã chia sẵn (BTXRD_split/train|val|test/{images,Annotations}) -> giữ nguyên cách chia
PRESPLIT = all(os.path.isdir(os.path.join(src, k, "Annotations")) for k in ("train", "val", "test"))
if PRESPLIT:
    part = {k: load_cases(os.path.join(src, k)) for k in ("train", "val", "test")}
    cases = part["train"] + part["val"] + part["test"]
    print("Dùng split có sẵn trong", src)
else:
    cases = load_cases(src)
print("annotated cases with polygon:", len(cases))

# loại u của cả ảnh: ác tính nếu có ít nhất 1 polygon ác tính
def case_type(stem, polys):
    return "malignant" if any(is_mal(stem, p["label"]) for p in polys) else "benign"

if not PRESPLIT:                           # chưa chia sẵn -> chia stratified theo từng loại u
    rnd = random.Random(seed)
    part = {"train": [], "val": [], "test": []}
    for t in ("benign", "malignant"):
        grp = [c for c in cases if case_type(c[0], c[3]) == t]
        rnd.shuffle(grp)
        n_te = int(len(grp) * r_te + 0.5)
        n_va = int(len(grp) * r_va + 0.5)
        part["test"] += grp[:n_te]
        part["val"] += grp[n_te:n_te + n_va]
        part["train"] += grp[n_te + n_va:]
    for k in ("test", "val"):              # bộ dữ liệu rất nhỏ: đảm bảo mỗi tập có ít nhất 1 ảnh
        if not part[k] and len(part["train"]) > 1:
            part[k].append(part["train"].pop())
splits = {"Tr": part["train"] + part["val"], "Ts": part["test"]}
for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    os.makedirs(os.path.join(out, d), exist_ok=True)

for split, items in splits.items():
    for stem, ip, ann, polys in items:
        img = ImageOps.exif_transpose(Image.open(ip)).convert("L")   # LabelMe cũng áp dụng EXIF orientation
        w, h = img.size
        sx = w / float(ann.get("imageWidth") or w)
        sy = h / float(ann.get("imageHeight") or h)
        mask = np.zeros((h, w), np.uint8)
        for c in (1, 2):   # vẽ ác tính sau để ưu tiên khi chồng lấn
            for s in polys:
                if cls(stem, s["label"]) == c:
                    pts = np.array([[x * sx, y * sy] for x, y in s["points"]], np.float64).round().astype(np.int32)
                    cv2.fillPoly(mask, [pts], c)
        img = np.array(img)
        if max_size and max(h, w) > max_size:
            r = max_size / max(h, w)
            nw, nh = int(round(w * r)), int(round(h * r))
            img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(os.path.join(out, f"images{split}", f"{stem}_0000.png"), img)
        cv2.imwrite(os.path.join(out, f"labels{split}", f"{stem}.png"), mask)

json.dump({"channel_names": {"0": "X-ray"}, "labels": labels, "numTraining": len(splits["Tr"]),
           "file_ending": ".png", "name": name,
           "description": "BTXRD bone tumor X-ray, polygons -> segmentation masks"},
          open(os.path.join(out, "dataset.json"), "w"), indent=2)
info = {k: sorted(c[0] for c in v) for k, v in part.items()}
info["type"] = {c[0]: case_type(c[0], c[3]) for c in cases}
json.dump(info, open(os.path.join(out, "split.json"), "w"), indent=1)
print("labels:", labels)
for k in ("train", "val", "test"):
    nb = sum(info["type"][c] == "benign" for c in info[k])
    print(f"{k:5s}: {len(info[k]):5d} ảnh ({len(info[k]) / len(cases):.1%})  | u lành {nb}, u ác {len(info[k]) - nb}")
"""
open(f"{TMP}/convert_btxrd.py", "w").write(convert_src)

if DATA_SOURCE == "btxrd":
    if BTXRD_DIR is None:
        # ưu tiên Kaggle Dataset (bản đầy đủ), sau đó tới data/BTXRD có sẵn trong repo GitHub
        # 1) thư mục đã chia sẵn train/val/test (BTXRD_split), 2) thư mục BTXRD thường (images/ + Annotations/)
        presplit = [os.path.dirname(os.path.dirname(p)) for p in
                    glob.glob("/kaggle/input/**/train/Annotations", recursive=True) + glob.glob(f"{REPO}/data/**/train/Annotations", recursive=True)]
        presplit = [d for d in presplit if all(os.path.isdir(f"{d}/{k}/images") for k in ("train", "val", "test"))]
        cands = glob.glob("/kaggle/input/**/Annotations", recursive=True) + glob.glob(f"{REPO}/data/**/Annotations", recursive=True)
        plain = [os.path.dirname(p) for p in cands if os.path.isdir(os.path.join(os.path.dirname(p), "images"))
                 and os.path.basename(os.path.dirname(p)) not in ("train", "val", "test")]
        found = presplit + plain
        assert found, "Không tìm thấy dữ liệu BTXRD trong /kaggle/input hoặc repo/data"
        BTXRD_DIR = found[0]
    print("BTXRD_DIR =", BTXRD_DIR)
    sig = json.dumps({"src": BTXRD_DIR, "mode": LABEL_MODE, "max": MAX_SIZE, "ratio": SPLIT_RATIO, "seed": SEED})
    sig_file = f"{RAW_DS}/convert_cfg.json"
    if not (os.path.isfile(sig_file) and open(sig_file).read() == sig):
        print("Chuyển đổi dữ liệu (cấu hình mới hoặc chưa có)...")
        !rm -rf {RAW_DS}/* {TMP}/nnUNet_preprocessed/{DATASET_NAME}
        ratios = ",".join(str(r) for r in SPLIT_RATIO)
        !python {TMP}/convert_btxrd.py "{BTXRD_DIR}" "{RAW_DS}" {LABEL_MODE} {MAX_SIZE} {ratios} {SEED} {DATASET_NAME}
        open(sig_file, "w").write(sig)
    else:
        print("Dữ liệu đã chuyển đổi với cùng cấu hình -> bỏ qua")
else:
    # Dữ liệu đã ở định dạng nnU-Net (vd Dataset704_Endovis17 của U-Mamba): DATASET_ID/NAME phải khớp tên thư mục
    !cp -rn {NNUNET_RAW_DIR}/. {RAW_DS}/

!cat {RAW_DS}/dataset.json
for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    print(d, len(glob.glob(f"{RAW_DS}/{d}/*")))

## 6. Plan & preprocess (giống README gốc)

In [ ]:
os.chdir(REPO)
PP_DIR = f"{TMP}/nnUNet_preprocessed/{DATASET_NAME}"
plans_path = f"{PP_DIR}/nnUNetPlans.json"
if not os.path.isfile(plans_path):
    !nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 2d -np 4 --verify_dataset_integrity

# Dùng đúng tập train/val 80:15 thay vì 5-fold ngẫu nhiên của nnU-Net (fold 0 = split của mình)
split_file = f"{RAW_DS}/split.json"
if os.path.isfile(split_file):
    sp = json.load(open(split_file))
    json.dump([{"train": sp["train"], "val": sp["val"]}], open(f"{PP_DIR}/splits_final.json", "w"), indent=1)
    print(f"splits_final.json: train={len(sp['train'])}, val={len(sp['val'])} | test (imagesTs)={len(sp['test'])}")

plans = json.load(open(plans_path))
if BATCH_SIZE:
    plans["configurations"]["2d"]["batch_size"] = int(BATCH_SIZE)
    json.dump(plans, open(plans_path, "w"), indent=2)
cfg = plans["configurations"]["2d"]
print("patch_size:", cfg["patch_size"], "| batch_size:", cfg["batch_size"])

## 7. Train + Validation
- Gọi `nnUNetv2_train <ID> 2d <FOLD> -tr <TRAINER>` giống script gốc.
- Với `LABEL_MODE="benign_malignant"`, model có **3 kênh đầu ra** (nền / u lành / u ác), nên vừa phân đoạn vừa phân loại loại u cho từng pixel.
- nnU-Net đọc `splits_final.json`: fold 0 = **80% train / 15% val** đã chia ở bước 5. Trong log sẽ có dòng `Using splits from existing split file`.
- Val được dùng để theo dõi Pseudo Dice từng epoch và chọn `checkpoint_best`. Khi train xong, nnU-Net chạy **validation** đầy đủ trên tập val và ghi kết quả vào `fold_0/validation/summary.json`.
- Nếu đã có `checkpoint_latest.pth` (phiên trước bị dừng, hoặc đã đặt `RESUME_DIR`), notebook sẽ tự thêm `--c` để train tiếp.

In [ ]:
os.chdir(REPO)
OUT_DIR  = f"{REPO}/data/nnUNet_results/{DATASET_NAME}/{TRAINER}__nnUNetPlans__2d"
FOLD_DIR = f"{OUT_DIR}/fold_{FOLD}"
resume = "--c" if os.path.isfile(f"{FOLD_DIR}/checkpoint_latest.pth") or os.path.isfile(f"{FOLD_DIR}/checkpoint_final.pth") else ""
print("resume flag:", resume or "(train mới)")
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_train {DATASET_ID} 2d {FOLD} -tr {TRAINER} {resume}

In [ ]:
# Kết quả validation & đường cong huấn luyện
import json
from IPython.display import Image as IPImage, display
summ = f"{FOLD_DIR}/validation/summary.json"
if os.path.isfile(summ):
    s = json.load(open(summ))
    print("Validation mean Dice (foreground):", round(s["foreground_mean"]["Dice"], 4))
    for k, v in s["mean"].items():
        print(f"  class {k}: Dice={v['Dice']:.4f}  IoU={v['IoU']:.4f}")
else:
    print("Chưa có validation/summary.json (train chưa xong?)")
if os.path.isfile(f"{FOLD_DIR}/progress.png"):
    display(IPImage(f"{FOLD_DIR}/progress.png"))

## 8. Inference trên tập test (giống script gốc: `--disable_tta`)

In [ ]:
os.chdir(REPO)
CKPT = next(c for c in ["checkpoint_final.pth", "checkpoint_best.pth", "checkpoint_latest.pth"]
            if os.path.isfile(f"{FOLD_DIR}/{c}"))
PRED_DIR = f"{OUT_DIR}/pred_results_fold{FOLD}"
print("checkpoint:", CKPT)
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_predict -i "{RAW_DS}/imagesTs" -o "{PRED_DIR}" -d {DATASET_ID} -c 2d -tr {TRAINER} -f {FOLD} -chk {CKPT} --disable_tta

## 9. Đánh giá DSC & NSD (dùng script đánh giá gốc cho ảnh PNG 2D)

In [ ]:
os.chdir(REPO)
!python evaluation/endoscopy_DSC_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_DSC_fold{FOLD}.csv"
!python evaluation/endoscopy_NSD_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_NSD_fold{FOLD}.csv"

## 9b. Đánh giá theo từng loại u: phân đoạn + phân loại u lành / u ác
Model dự đoán mỗi pixel thuộc **nền (0)**, **u lành (1)** hay **u ác (2)**. Ô này đánh giá kết quả ở hai mức:
- **Mức pixel (phân đoạn):** Dice và IoU **riêng cho từng lớp** trên tập test, tính trên toàn bộ pixel của tập test (gộp tất cả ảnh) và trung bình theo từng ảnh.
- **Mức ảnh (phân loại):** loại u của một ảnh là lớp chiếm nhiều pixel nhất trong vùng u dự đoán. Kết quả gồm ma trận nhầm lẫn, accuracy, và **độ nhạy (sensitivity) với u ác**, tức tỉ lệ u ác được phát hiện đúng; đây là chỉ số quan trọng nhất về mặt lâm sàng.

In [ ]:
import numpy as np, cv2, pandas as pd

CLS = {1: "u lành", 2: "u ác"}
def img_type(m):
    # loại u của ảnh = lớp chiếm nhiều pixel nhất trong vùng u; 0 = không tìm thấy u
    c1, c2 = int((m == 1).sum()), int((m == 2).sum())
    return 0 if c1 + c2 == 0 else (2 if c2 > c1 else 1)

rows, tot = [], {c: [0, 0, 0] for c in CLS}          # tot[c] = [TP, FP, FN] cộng dồn mọi pixel
for n in sorted(f for f in os.listdir(PRED_DIR) if f.endswith(".png")):
    gt = cv2.imread(f"{RAW_DS}/labelsTs/{n}", cv2.IMREAD_UNCHANGED)
    pr = cv2.imread(f"{PRED_DIR}/{n}", cv2.IMREAD_UNCHANGED)
    r = {"case": n[:-4], "gt_type": img_type(gt), "pred_type": img_type(pr)}
    for c in CLS:
        g, p = gt == c, pr == c
        tp, fp, fn = int((g & p).sum()), int((~g & p).sum()), int((g & ~p).sum())
        for j, v in enumerate((tp, fp, fn)): tot[c][j] += v
        # Dice theo ảnh: chỉ tính khi lớp có trong GT hoặc trong dự đoán
        r[f"Dice_{c}"] = 2 * tp / (2 * tp + fp + fn) if (tp + fp + fn) else np.nan
    r["Dice_tumor"] = (lambda g, p: 2 * (g & p).sum() / max(g.sum() + p.sum(), 1))(gt > 0, pr > 0)
    rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/metric_per_class_fold{FOLD}.csv", index=False)

print("=== Phân đoạn (tập test,", len(df), "ảnh) ===")
for c, name in CLS.items():
    tp, fp, fn = tot[c]
    print(f"{name:7s}: Dice(gộp)={2*tp/max(2*tp+fp+fn,1):.4f}  IoU(gộp)={tp/max(tp+fp+fn,1):.4f}  "
          f"Dice TB theo ảnh={df[f'Dice_{c}'].mean():.4f}")
print(f"khối u chung (bỏ qua loại): Dice TB theo ảnh={df['Dice_tumor'].mean():.4f}")

print("\n=== Phân loại mức ảnh ===")
labels_txt = {0: "không thấy u", 1: "u lành", 2: "u ác"}
cm = pd.crosstab(df["gt_type"].map(labels_txt), df["pred_type"].map(labels_txt),
                 rownames=["Thực tế"], colnames=["Dự đoán"], dropna=False)
display(cm)
acc = (df["gt_type"] == df["pred_type"]).mean()
mal = df[df["gt_type"] == 2]; ben = df[df["gt_type"] == 1]
print(f"Accuracy: {acc:.3f}")
if len(mal): print(f"Độ nhạy với u ác (sensitivity): {(mal['pred_type'] == 2).mean():.3f}  ({(mal['pred_type'] == 2).sum()}/{len(mal)})")
if len(ben): print(f"Độ đặc hiệu (specificity, u lành đoán đúng): {(ben['pred_type'] == 1).mean():.3f}  ({(ben['pred_type'] == 1).sum()}/{len(ben)})")

## 10. Trực quan hoá: u lành (xanh) và u ác (đỏ)

In [ ]:
import random
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
%matplotlib inline

CMAP = ListedColormap([(0, 0, 0, 0), (0.1, 0.8, 0.2, 1), (0.95, 0.15, 0.1, 1)])   # nền trong suốt / lành xanh / ác đỏ
TYPE = {0: "không thấy u", 1: "U LÀNH", 2: "U ÁC"}

def show(names, title=""):
    k = len(names)
    if not k: return
    fig, ax = plt.subplots(k, 3, figsize=(13, 4.2 * k), squeeze=False)
    for i, n in enumerate(names):
        img = cv2.imread(f"{RAW_DS}/imagesTs/{n}_0000.png", cv2.IMREAD_GRAYSCALE)
        gt  = cv2.imread(f"{RAW_DS}/labelsTs/{n}.png", cv2.IMREAD_UNCHANGED)
        pr  = cv2.imread(f"{PRED_DIR}/{n}.png", cv2.IMREAD_UNCHANGED)
        tg, tp = img_type(gt), img_type(pr)
        ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title(n)
        ax[i, 1].imshow(img, cmap="gray"); ax[i, 1].imshow(gt, cmap=CMAP, vmin=0, vmax=2, alpha=0.55, interpolation="nearest")
        ax[i, 1].set_title(f"Ground truth: {TYPE[tg]}")
        ax[i, 2].imshow(img, cmap="gray"); ax[i, 2].imshow(pr, cmap=CMAP, vmin=0, vmax=2, alpha=0.55, interpolation="nearest")
        ax[i, 2].set_title(f"Dự đoán: {TYPE[tp]}  " + ("✓" if tg == tp else "✗"), color="green" if tg == tp else "red")
        for a in ax[i]: a.axis("off")
    fig.legend(handles=[Patch(color=CMAP(1), label="u lành"), Patch(color=CMAP(2), label="u ác")],
               loc="upper right", fontsize=12)
    fig.suptitle(title, fontsize=14); plt.tight_layout(); plt.show()

rnd = random.Random(0)
mal = df[df.gt_type == 2].case.tolist(); ben = df[df.gt_type == 1].case.tolist()
wrong = df[df.gt_type != df.pred_type].case.tolist()
show(rnd.sample(mal, min(2, len(mal))) + rnd.sample(ben, min(2, len(ben))), "Ví dụ u ác & u lành")
show(wrong[:4], f"Các ảnh phân loại sai ({len(wrong)} ảnh)")

## 11. Lưu kết quả
Mọi thứ trong `Swin-UMamba/data/nnUNet_results/` (checkpoint, log, validation, dự đoán, file CSV metric) nằm trong `/kaggle/working` và sẽ được lưu cùng version notebook. Để train tiếp ở phiên sau: thêm output của version này làm input, rồi đặt `RESUME_DIR` trỏ tới thư mục `nnUNet_results` trong đó.

In [ ]:
!du -sh {REPO}/data/nnUNet_results
!ls -la {FOLD_DIR}